# 01 — LoRA paper summary (in my own words)

**Paper:** Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, ICLR 2022 (arXiv:2106.09685).

This notebook is the reading notes for the reproduction. No training happens here — the point is to show I understood the method *before* running it. If a line below is wrong, the reproduction built on it would be wrong too, so these notes are part of the work, not a formality.

## The problem

Fine-tuning a large pre-trained model updates **every** weight. For a model with billions of parameters that means storing optimizer state for billions of parameters, and a separate full copy of the model for every downstream task. That is expensive and wasteful when the task only needs a small change to the model's behaviour.

## The idea

LoRA freezes the pre-trained weights $W_0$ and learns only a low-rank update:

$$W = W_0 + \Delta W, \qquad \Delta W = BA$$

where $B$ is $d \times r$, $A$ is $r \times k$, and the rank $r$ is much smaller than $d$ and $k$. At the start of training $A$ is random Gaussian and $B$ is zero, so $\Delta W = 0$ — the model behaves exactly like the pre-trained model until training moves it. During inference the adapter can be merged back into $W_0$, so there is no extra latency.

The paper applies this to the attention weight matrices (query and value) of Transformers. The key claim: a low-rank update is enough, because the *change* a task needs has low "intrinsic rank" even when the model itself is huge.

## What the paper actually shows

- On GPT-3 (175B), LoRA with $r=4$ or $r=8$ matches full fine-tuning on several benchmarks while training ~10,000x fewer parameters.
- Larger rank helps up to a point, then plateaus — which is exactly what the ablation in this repo tests at small scale.
- LoRA is **not** a new architecture. It is a training method: same model, fewer trainable weights.

## How this repo tests it (small scale)

| Paper | This reproduction |
|---|---|
| GPT-3 / RoBERTa-large | DistilBERT (~66M) — runnable on free Colab |
| Many tasks | One task: SST-2 sentiment classification |
| Rank up to 64 | Rank 4 / 8 / 16 |

This does **not** replicate the paper's numbers. It tests whether the *method* behaves as claimed — rank trades parameters for accuracy, with diminishing returns — on a model I can actually run. That distinction is stated in the report too.

## Sanity check: how many parameters does LoRA actually add?

DistilBERT hidden size is 768. Each attention layer has a `q_lin` and a `v_lin` matrix of shape 768×768. A LoRA adapter for one such matrix adds $r \times (768 + 768)$ parameters. Run the cell below and compare with the counts `train.py` prints — they must agree.

In [ ]:
# LoRA parameter count, by hand — must match what train.py reports.
d = 768            # DistilBERT hidden size
n_layers = 6       # DistilBERT has 6 transformer layers
target_modules = 2 # q_lin and v_lin per layer
for r in [4, 8, 16]:
    per_matrix = r * (d + d)
    total = per_matrix * target_modules * n_layers
    print(f"r={r:2d}: {total:,} trainable params ({total / 66_000_000 * 100:.3f}% of ~66M)")
print()
print("Full fine-tuning updates ~66M. LoRA r=8 updates ~0.03% of that.")
print("If train.py ever prints a very different number, the config is wrong — stop and check.")

## My one-paragraph summary (write this yourself after reading)

> *LoRA freezes a pre-trained model and learns only a small low-rank correction to its attention weights. Because the correction is low-rank, it needs very few trainable parameters, can be merged back for inference at no extra cost, and — the paper claims — still matches full fine-tuning. This project checks that claim on DistilBERT + SST-2 and measures how the trade-off moves with rank.*

**Next:** notebook 02 runs the full fine-tuning baseline that everything else is compared against.